# Component analysis — review

Renders one **cell** of a component-analysis results directory: the head table,
the attention profiles, direct logit attribution and the logit lens, the
knockout ranking, path patching, and the segmentation and sample the causal
analyses ran over.

A cell is `<output-dir>/<circuit label>/<task label>/`, written by
`analyze_components.py`. Everything above the live section reads **only** the
JSON and CSV in that directory, so it runs on the login node with no GPU and no
model. An analysis the cell does not contain renders a note; nothing raises.

The last section is the drill-down. It is skipped entirely unless `LIVE` is
`True` in the first cell, because it loads the 8B model onto a GPU — which
means it must run inside a Slurm job on `gpuA100x4` or `gpuA40x4`, never on a
login node.

Every decision this notebook makes about a results directory lives in
`component_analysis/review.py` and is tested in `tests/test_review.py`; the
cells below are renderer calls.

In [ ]:
# ============================ CONFIGURATION =================================

#: One cell of a results directory: <output-dir>/<circuit label>/<task label>.
RESULTS_DIR = "outputs/_analyses/ca06-paired2/delta__head/sdr"

#: False everywhere except inside a Slurm GPU job. See the live section.
LIVE = False

#: The repository root, so `import component_analysis` works from anywhere.
REPO_ROOT = "/projects/beto/swong2/Alignment/Deception-Circuit"

#: Which instance, head and path the live section drills into. None takes the
#: first instance of the cell's own causal sample.
LIVE_INSTANCE_ID = None
LIVE_HEAD = (14, 22)               # (layer, index)
LIVE_SENDER = ("head", 14, 22)     # any Component, in the circuit or not
LIVE_RECEIVER = ("q", 24, 25)      # ("q"|"k"|"v"|"mlp_in", layer, index)

# ============================================================================

import os
import sys

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from component_analysis import review

print("results directory:", os.path.abspath(RESULTS_DIR))
print("exists:", os.path.isdir(RESULTS_DIR))
print("LIVE:", LIVE)

In [ ]:
# The renderers. Each one maps a shape from `review` onto one plotly figure or
# one table; the dispatch below is the only branch, and `Missing` prints its
# note instead of raising.

import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display


def heading(view):
    display(Markdown(f"**{view.title}**  \n{view.subtitle}"))
    for note in getattr(view, "notes", None) or []:
        display(Markdown(f"> {note}"))


def show_heatmap(view):
    heading(view)
    figure = go.Figure(go.Heatmap(
        z=view.values, x=view.columns, y=view.rows,
        colorscale="RdBu", zmid=0,
        zmin=None if view.extent is None else -view.extent,
        zmax=view.extent, hoverongaps=False))
    figure.update_layout(height=max(320, 24 * len(view.rows) + 180),
                         yaxis=dict(autorange="reversed"),
                         margin=dict(l=160, t=40, b=140))
    figure.show()


def show_bars(view):
    heading(view)
    figure = go.Figure(go.Bar(
        x=view.labels, y=view.values, text=view.hover,
        error_y=None if view.errors is None else dict(type="data",
                                                      array=view.errors)))
    figure.update_layout(height=420, margin=dict(t=40, b=140),
                         xaxis=dict(type="category"))
    figure.show()


def show_lines(view):
    heading(view)
    figure = go.Figure()
    for series in view.series:
        figure.add_trace(go.Scatter(x=view.x, y=series.y, mode="lines+markers",
                                    name=series.name))
    figure.add_hline(y=0, line_dash="dot")
    figure.update_layout(height=460, xaxis_title=view.x_title,
                         margin=dict(t=40))
    figure.show()


def show_scatter(view):
    heading(view)
    figure = go.Figure(go.Scatter(
        x=view.x, y=view.y, text=view.labels, mode="markers+text",
        textposition="top center",
        error_x=None if view.x_se is None else dict(type="data", array=view.x_se),
        error_y=None if view.y_se is None else dict(type="data", array=view.y_se)))
    limit = max([abs(one) for one in view.x + view.y if one is not None] or [1])
    figure.add_trace(go.Scatter(x=[-limit, limit], y=[-limit, limit],
                                mode="lines", name="y = x",
                                line=dict(dash="dot")))
    figure.update_layout(height=520, xaxis_title=view.x_title,
                         yaxis_title=view.y_title, margin=dict(t=40))
    figure.show()


def show_table(view):
    heading(view)
    display(pd.DataFrame(view.rows, columns=view.columns))


def render(view):
    if isinstance(view, review.Missing):
        display(Markdown(f"_{view.note}_"))
    elif isinstance(view, review.Group):
        heading(view)
        for item in view.items:
            render(item)
    elif isinstance(view, review.Heatmap):
        show_heatmap(view)
    elif isinstance(view, review.Bars):
        show_bars(view)
    elif isinstance(view, review.Lines):
        show_lines(view)
    elif isinstance(view, review.Scatter):
        show_scatter(view)
    elif isinstance(view, review.Table):
        show_table(view)
    else:
        raise TypeError(f"no renderer for {type(view).__name__}")

## 1. Head table

One row per Component — attention head or MLP block — keyed by layer and index,
with every analysis's columns namespaced by its own name. The Component Role
triple is brought to the front when it is there; until ticket 10's proposal
step has run on this cell, a note says so instead.

In [ ]:
HEAD_TABLE = review.head_table_view(RESULTS_DIR)
render(HEAD_TABLE)

## 2. Attention patterns by Segment

Attention mass out of the query position (the Readout by default), summed per
Segment and averaged over instances. One heatmap per level of the split and
per condition: under pressure, under the Matched Honest Instruction (paired),
and on the honest twin (unpaired, descriptive).

In [ ]:
ATTENTION = review.attention_views(RESULTS_DIR)
render(ATTENTION)

In [ ]:
render(review.top_heads_view(RESULTS_DIR))

In [ ]:
# The same first profile in circuitsvis: each Segment is coloured by how much
# of the head's attention it takes, and the buttons switch between heads.

if ATTENTION:
    import torch
    from circuitsvis.tokens import colored_tokens_multi

    FIRST = ATTENTION.items[0]
    segments, masses, heads = review.coloured_tokens(FIRST)
    display(Markdown(f"**{FIRST.title}** — {FIRST.subtitle}"))
    display(colored_tokens_multi(segments, torch.tensor(masses), heads))
else:
    display(Markdown(f"_{ATTENTION.note}_"))

## 3. Direct logit attribution and the logit lens

Each Component's direct contribution to logit(Target) minus logit(Distractor)
at the Readout, read in float32 through the model's own final norm. The paired
scatter puts each Component under pressure against itself under the Matched
Honest Instruction on the same instances; a point off the diagonal writes
something different when it knows it is lying.

In [ ]:
render(review.dla_heatmap(RESULTS_DIR))

In [ ]:
render(review.dla_view(RESULTS_DIR))

In [ ]:
render(review.dla_scatter_view(RESULTS_DIR))

In [ ]:
render(review.logit_lens_views(RESULTS_DIR))

## 4. Knockout

Each Component ablated on its own by interchange with the item-swap stream,
ranked by how much the deceptive answer loses. **A positive drop means the
Component helps deception** — the opposite of path patching's sign, which is
why each section prints its own convention.

In [ ]:
render(review.knockout_view(RESULTS_DIR))

In [ ]:
render(review.family_knockout_view(RESULTS_DIR))

## 5. Path patching

Senders down the rows, receivers across the columns, each cell the change in
the Readout metric when that path alone carries the counterfactual, normalized
by the instance's clean-minus-corrupt gap. **Negative means the path carries
the counterfactual.** Grey cells are pairs where the sender is not strictly
upstream of the receiver.

In [ ]:
render(review.path_patching_views(RESULTS_DIR))

In [ ]:
render(review.sweep_view(RESULTS_DIR))

## 6. Segmentation and sample

What the Segments actually were on this split, and which instances the causal
analyses ran over. Read these before reading any plot above: a heatmap column
means nothing without the span behind it.

In [ ]:
render(review.segmentation_view(RESULTS_DIR))

In [ ]:
render(review.sample_view(RESULTS_DIR))

## 7. Live drill-down

**Everything below needs a GPU.** It is skipped when `LIVE` is `False`, which
is the default and the only correct value on a login node. Run it inside a
Slurm job on `gpuA100x4` or `gpuA40x4` (account `beto-delta-gpu`), the same
partition `scripts/analyze_components.sbatch` submits to.

The model is loaded once, through `component_analysis.hooks`, and the three
drill-downs call the same library functions the CLI calls, with the same
signatures — there is no second nnsight path.

In [ ]:
if LIVE:
    import torch

    from component_analysis import hooks
    from component_analysis import instances as ca_instances

    if not torch.cuda.is_available():
        raise RuntimeError(
            "LIVE=True but no GPU is visible. This section loads an 8B model; "
            "run the notebook inside a Slurm job on gpuA100x4 or gpuA40x4, "
            "never on a login node.")

    MODEL_NAME = review.model_name_of(HEAD_TABLE.meta)
    HF_TOKEN = os.environ.get("HF_TOKEN")
    MODEL = hooks.load_analysis_model(MODEL_NAME, device="cuda",
                                      hf_token=HF_TOKEN)
    TOKENIZER = ca_instances.load_analysis_tokenizer(MODEL_NAME,
                                                     hf_token=HF_TOKEN)
    DEVICE, DTYPE = hooks.device_and_dtype(MODEL)
    print(hooks.describe(MODEL))
else:
    print("LIVE is False: no model is loaded and the rest of this section "
          "does nothing.")

In [ ]:
# One instance of this cell's own causal sample, loaded through the CLI's
# loader so the prompts, the Readout and both counterfactual streams are the
# ones the analyses used.

if LIVE:
    from component_analysis import dataloaders

    SAMPLE = review.read_json(os.path.join(RESULTS_DIR, review.SAMPLE_JSON))
    INSTANCE_ID = LIVE_INSTANCE_ID or SAMPLE["instance_ids"][0]
    TASK_SPEC = review.task_spec_from_meta(HEAD_TABLE.meta)
    BATCH = next(iter(dataloaders.batches_for_task(
        TASK_SPEC, TOKENIZER, ids=[INSTANCE_ID], batch_size=1)))
    TOKENS = TOKENIZER.convert_ids_to_tokens(BATCH.input_ids[0].tolist())
    print("instance", INSTANCE_ID, "·", len(TOKENS), "tokens ·",
          "Readout at", int(BATCH.readout[0]))

### 7a. One head's attention on one instance

`hooks.attention_weights` with the CLI's own signature: the rows out of the
Readout position for one head, over the real tokens rather than over Segments.

In [ ]:
if LIVE:
    from circuitsvis.tokens import colored_tokens

    LAYER, HEAD = LIVE_HEAD
    ROWS = hooks.attention_weights(
        MODEL, BATCH.input_ids.to(DEVICE), BATCH.attention_mask.to(DEVICE),
        layer=LAYER, heads=[HEAD],
        query_positions=BATCH.readout.to(DEVICE).unsqueeze(-1))
    display(Markdown(f"**L{LAYER}H{HEAD}** at the Readout, instance "
                     f"{INSTANCE_ID}"))
    display(colored_tokens(TOKENS, ROWS[0, 0, 0].float().cpu()))

### 7b. Direct logit attribution on a hand-typed prompt

`dla.measure_batch`, the function the DLA analysis runs per batch. Type any
prompt and name the two answers; the attribution is over this cell's
Components plus the embedding, in float32 through the real final norm.

In [ ]:
if LIVE:
    from component_analysis import dla

    LIVE_PROMPT = TOKENIZER.decode(BATCH.input_ids[0][
        BATCH.attention_mask[0].bool()])
    LIVE_TARGET = TOKENIZER.decode([int(BATCH.target_token[0])])
    LIVE_DISTRACTOR = TOKENIZER.decode([int(BATCH.distractor_token[0])])

    encoded = TOKENIZER(LIVE_PROMPT, return_tensors="pt",
                        add_special_tokens=False)
    live_ids = encoded["input_ids"].to(DEVICE)
    live_mask = encoded["attention_mask"].to(DEVICE)
    live_readout = (live_mask.sum(-1).long() - 1).to(DEVICE)
    target = torch.tensor(
        TOKENIZER.encode(LIVE_TARGET, add_special_tokens=False)[-1:],
        device=DEVICE)
    distractor = torch.tensor(
        TOKENIZER.encode(LIVE_DISTRACTOR, add_special_tokens=False)[-1:],
        device=DEVICE)

    KEYS = [(row["kind"], row["layer"], row["index"]) for row in HEAD_TABLE.rows]
    per_component, lens, total = dla.measure_batch(
        MODEL, KEYS, live_ids, live_mask, live_readout, target, distractor)

    display(pd.DataFrame(
        {"component": [review.component_label(key) for key in KEYS],
         "dla": per_component[0].tolist()}).sort_values("dla"))
    print(f"target {LIVE_TARGET!r} minus distractor {LIVE_DISTRACTOR!r} = "
          f"{float(total[0]):+.4f}")

### 7c. One off-grid patch

`path_patching.capture_streams` then `patched_readout` then
`normalized_effect` — the three calls a grid cell is made of, run once for a
sender and receiver of your choosing. Neither has to be in the circuit, which
is what makes this off-grid.

In [ ]:
if LIVE:
    from component_analysis import path_patching

    STREAMS = path_patching.capture_streams(
        MODEL, BATCH, senders=[LIVE_SENDER],
        counterfactual=path_patching.ITEM_SWAP)
    if STREAMS is None:
        print("this instance has no counterpart under that counterfactual")
    else:
        PATCHED = path_patching.patched_readout(
            MODEL, STREAMS, senders=[LIVE_SENDER], receivers=[LIVE_RECEIVER])
        EFFECT = path_patching.normalized_effect(STREAMS, PATCHED)
        print(f"{path_patching.label(LIVE_SENDER)} -> "
              f"{path_patching.label(LIVE_RECEIVER)}: "
              f"{float(EFFECT[0]):+.4f}")
        print(path_patching.metric_meta(STREAMS.counterfactual)["sign"])